In [1]:
import pandas as pd
import lbpscreen as lbp
from pathlib import Path

MODEL_DIR = Path(r"C:\Users\IPRDC\lbp-screen\data\processed\core_models")
FILE_MAP = {
    "E_faecalis_MMH594":    "Enterococcus_faecalis_MMH594_AGORA2_v2.01.xml",
    "E_lenta_DSM2243":      "Eggerthella_lenta_DSM_2243_AGORA2_v2.01.xml",
    "E_coli_Nissle1917":    "Escherichia_coli_Nissle_1917_AGORA2_v2.01.xml",
    "A_muciniphila_BAA835": "Akkermansia_muciniphila_ATCC_BAA_835_AGORA2_v2.01.xml",
}
models = lbp.load_models(MODEL_DIR, FILE_MAP)

In [3]:
# How many exchanges allow uptake? A negative lower bound means the nutrient may enter the cell
rows = []
for label, m in models.items():
    ex = m.exchanges
    rows.append({
        "strain": label,
        "exchanges": len(ex),
        "open_for_uptake": sum(r.lower_bound < 0 for r in ex),
        "open_for_secretion": sum(r.upper_bound > 0 for r in ex),
        "growth_now": m.slim_optimize(),
    })
pd.DataFrame(rows).set_index("strain")

,exchanges,open_for_uptake,open_for_secretion,growth_now
strain,,,,
E_faecalis_MMH594,173,173,173,0.0
E_lenta_DSM2243,121,121,121,0.0
E_coli_Nissle1917,569,569,569,0.0
A_muciniphila_BAA835,118,118,118,0.0


In [4]:
# Current medium of one model
nissle = models["E_coli_Nissle1917"]
medium = pd.Series(nissle.medium, name="max_uptake").sort_values(ascending=False)
print(len(medium), "nutrients open for uptake")
medium.head(20)

569 nutrients open for uptake


EX_5mthf_e_           1000.0
EX_ala_D_e_           1000.0
EX_andrstrnglc_e_     1000.0
EX_bildglcur_e_       1000.0
EX_bilglcur_e_        1000.0
EX_chol_e_            1000.0
EX_cholate_e_         1000.0
EX_crn_e_             1000.0
EX_dgchol_e_          1000.0
EX_estradiol_e_       1000.0
EX_estradiolglc_e_    1000.0
EX_estriolglc_e_      1000.0
EX_estroneglc_e_      1000.0
EX_gchola_e_          1000.0
EX_gsn_e_             1000.0
EX_hdca_e_            1000.0
EX_lac_D_e_           1000.0
EX_meoh_e_            1000.0
EX_ncam_e_            1000.0
EX_ocdca_e_           1000.0
Name: max_uptake, dtype: float64

In [6]:
# What does the biomass reaction need?
for label, m in models.items():
    obj_id = lbp.model_summary({label: m}).loc[label, "objective"]
    bio = m.reactions.get_by_id(obj_id)
    print(f"{label}: {obj_id}, bounds={bio.bounds}, "
          f"consumes {len(bio.reactants)} metabolites, produces {len(bio.products)}")

E_faecalis_MMH594: bio1, bounds=(0.0, 1000.0), consumes 91 metabolites, produces 9
E_lenta_DSM2243: biomass467, bounds=(0.0, 1000.0), consumes 80 metabolites, produces 9
E_coli_Nissle1917: bio1, bounds=(0.0, 1000.0), consumes 78 metabolites, produces 9
A_muciniphila_BAA835: biomass345, bounds=(0.0, 1000.0), consumes 78 metabolites, produces 9


In [20]:
for label, m in models.items():
    with m:
        for r in m.exchanges:
            r.lower_bound = -1000
        print(f"{label:22s} growth with everything available: {m.slim_optimize():.3f}")

E_faecalis_MMH594      growth with everything available: 0.000
E_lenta_DSM2243        growth with everything available: 0.000
E_coli_Nissle1917      growth with everything available: 0.000
A_muciniphila_BAA835   growth with everything available: 0.000


In [21]:
diet = lbp.load_diet(Path(r"C:\Users\IPRDC\lbp-screen\data\raw\diets\high-fiber.csv"))
for label, m in models.items():
    with m:
        missing = lbp.apply_medium(m, diet)
        print(f"{label:22s} growth on high-fiber: {m.slim_optimize():.3f} "
              f"({len(missing)} diet items not in model)")

E_faecalis_MMH594      growth on high-fiber: 0.000 (41 diet items not in model)
E_lenta_DSM2243        growth on high-fiber: 0.000 (68 diet items not in model)
E_coli_Nissle1917      growth on high-fiber: 0.000 (37 diet items not in model)
A_muciniphila_BAA835   growth on high-fiber: 0.000 (62 diet items not in model)


In [23]:
sbml = lbp.read_model(MODEL_DIR / FILE_MAP["E_coli_Nissle1917"])
js = lbp.read_model(Path(r"C:\Users\IPRDC\lbp-screen\data\raw\agora2_json\Escherichia_coli_Nissle_1917_AGORA2_v2.01.json"))
lbp.model_summary({"sbml": sbml, "json": js})

,reactions,metabolites,genes,exchanges,objective,format
strain,,,,,,
sbml,3574,3115,1457,569,bio1,sbml
json,3574,3115,1457,569,bio1,json


In [24]:
def compare_ids(a, b, attr):
    A = {x.id for x in getattr(a, attr)}
    B = {x.id for x in getattr(b, attr)}
    return {"shared": len(A & B), "only_sbml": sorted(A - B), "only_json": sorted(B - A)}

for attr in ["reactions", "metabolites", "genes"]:
    c = compare_ids(sbml, js, attr)
    print(f"\n{attr.upper()}: shared={c['shared']}, "
          f"only_sbml={len(c['only_sbml'])}, only_json={len(c['only_json'])}")
    print("  SBML examples:", c["only_sbml"][:8])
    print("  JSON examples:", c["only_json"][:8])


REACTIONS: shared=3000, only_sbml=574, only_json=574
  SBML examples: ['DM_dhptd_c_', 'DM_kdo2lipid4L_c_', 'DM_this_c_', 'EX_15dap_e_', 'EX_1hibup_S_e_', 'EX_1hibupglu_S_e_', 'EX_1hmdgluc_e_', 'EX_1ohmdz_e_']
  JSON examples: ['DM_dhptd(c)', 'DM_kdo2lipid4L(c)', 'DM_this[c]', 'EX_15dap[e]', 'EX_1hibup_S[e]', 'EX_1hibupglu_S[e]', 'EX_1hmdgluc[e]', 'EX_1ohmdz[e]']

METABOLITES: shared=3115, only_sbml=0, only_json=0
  SBML examples: []
  JSON examples: []

GENES: shared=768, only_sbml=689, only_json=689
  SBML examples: ['316435_3_1000_10_peg', '316435_3_1022_16_peg', '316435_3_1067_22_peg', '316435_3_1109_2_peg', '316435_3_1113_36_peg', '316435_3_115_13_peg', '316435_3_115_peg', '316435_3_123_6_peg']
  JSON examples: ['316435.3.1000_10.peg', '316435.3.1022_16.peg', '316435.3.1067_22.peg', '316435.3.1109_2.peg', '316435.3.1113_36.peg', '316435.3.115.peg', '316435.3.115_13.peg', '316435.3.123_6.peg']


In [25]:
for name, m in [("sbml", sbml), ("json", js)]:
    print(name, "default growth:", m.slim_optimize(),
          "| uptakes open:", sum(r.lower_bound < 0 for r in m.exchanges))

sbml default growth: 0.0 | uptakes open: 569
json default growth: 0.0 | uptakes open: 569
